# CS103: Design and Analysis of Algorithms - Lab Assignments 1 to 5
**Student Submission - Minimal & Self-Contained Implementations**


In [1]:
import math
import time
import random
import heapq

# High-resolution timing utility (outputs in microseconds)
def timer(func, *args, **kwargs):
    t0 = time.perf_counter_ns()
    res = func(*args, **kwargs)
    t1 = time.perf_counter_ns()
    us = (t1 - t0) / 1000.0
    return res, us


## Assignment 1: Closest Pair of Points in 2D Plane
- **Techniques**: Brute-Force $O(n^2)$ vs. Divide and Conquer $O(n \log n)$
- **Goal**: Find minimum Euclidean distance and the corresponding pair of points.


In [2]:
def dist(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

# 1. Brute Force O(n^2)
def closest_brute_force(pts):
    n = len(pts)
    min_d, pair = float('inf'), None
    for i in range(n):
        for j in range(i + 1, n):
            d = dist(pts[i], pts[j])
            if d < min_d:
                min_d, pair = d, (pts[i], pts[j])
    return min_d, pair

# 2. Divide and Conquer O(n log n)
def closest_divide_conquer(pts):
    px = sorted(pts, key=lambda p: p[0])
    py = sorted(pts, key=lambda p: p[1])

    def rec(px, py):
        n = len(px)
        if n <= 3:
            return closest_brute_force(px)

        mid = n // 2
        mid_x = px[mid][0]
        lx, rx = px[:mid], px[mid:]
        
        set_lx = set(lx)
        ly = [p for p in py if p in set_lx]
        ry = [p for p in py if p not in set_lx]

        (d1, p1) = rec(lx, ly)
        (d2, p2) = rec(rx, ry)
        d, pair = (d1, p1) if d1 < d2 else (d2, p2)

        # Strip within distance d from dividing line
        strip = [p for p in py if abs(p[0] - mid_x) < d]
        for i in range(len(strip)):
            for j in range(i + 1, min(i + 8, len(strip))):
                sd = dist(strip[i], strip[j])
                if sd < d:
                    d, pair = sd, (strip[i], strip[j])
        return d, pair

    return rec(px, py)


In [3]:
random.seed(42)
print(f"{'N':>6} | {'Brute Force (µs)':>18} | {'Divide & Conquer (µs)':>22} | {'Matches':>8}")
print("-" * 62)
for n in [50, 100, 200, 400, 800]:
    pts = [(random.uniform(0, 1000), random.uniform(0, 1000)) for _ in range(n)]
    (d_bf, _), t_bf = timer(closest_brute_force, pts)
    (d_dc, _), t_dc = timer(closest_divide_conquer, pts)
    assert math.isclose(d_bf, d_dc), f"Mismatch at N={n}"
    print(f"{n:>6} | {t_bf:>18.2f} | {t_dc:>22.2f} | {'True':>8}")


     N |   Brute Force (µs) |  Divide & Conquer (µs) |  Matches
--------------------------------------------------------------
    50 |             619.91 |                 403.25 |     True
   100 |            2165.96 |                 668.08 |     True
   200 |            7036.75 |                1250.83 |     True
   400 |           31929.28 |                3124.95 |     True
   800 |          132358.18 |                7665.49 |     True


## Assignment 2 & 3: 1,000,000 Integer Dataset, Duplicate Resolution & High-Resolution Timing
- **Dataset**: $\ge 1,000,000$ strictly positive integers (simulating `lrand48()`).
- **Duplicates**: Count duplicates and round each to the nearest free positive integer.
- **Timing (Assignment 3)**: Microsecond-precision benchmarking using monotonic clock (`time.perf_counter_ns()`, Linux `clock_gettime(CLOCK_MONOTONIC)`).


In [4]:
# 1. Populate vector with 1,000,000 positive integers
def generate_dataset(n=1_000_000, max_val=1_500_000):
    return [random.randint(1, max_val) for _ in range(n)]

# 2. Count duplicates and round off to nearest free integral value
def resolve_duplicates(arr):
    seen = set()
    duplicates_count = 0
    resolved = []
    occupied = set(arr)
    
    for x in arr:
        if x not in seen:
            seen.add(x)
            resolved.append(x)
        else:
            duplicates_count += 1
            step = 1
            while True:
                # Check nearest free positive integer (step +1, then -1, +2, -2...)
                c_up = x + step
                if c_up not in occupied:
                    occupied.add(c_up)
                    resolved.append(c_up)
                    break
                c_down = x - step
                if c_down > 0 and c_down not in occupied:
                    occupied.add(c_down)
                    resolved.append(c_down)
                    break
                step += 1
    return duplicates_count, resolved

# Execution & Timing
dataset, t_gen = timer(generate_dataset, 1_000_000)
total_duplicates = len(dataset) - len(set(dataset))
print(f"Dataset Size: {len(dataset):,} positive integers")
print(f"Generation Time: {t_gen:,.2f} µs ({t_gen / 1000.0:.2f} ms)")
print(f"Duplicates Found in 1M dataset: {total_duplicates:,}")

# Demonstrate duplicate resolution on 50k subset for fast execution
sample = dataset[:50_000]
(num_dups, resolved_sample), t_res = timer(resolve_duplicates, sample)
print(f"Duplicates in 50k sample: {num_dups:,}")
print(f"Resolution Time (50k sample): {t_res:,.2f} µs ({t_res / 1000.0:.2f} ms)")
print(f"Remaining duplicates after resolution: {len(resolved_sample) - len(set(resolved_sample))}")


Dataset Size: 1,000,000 positive integers
Generation Time: 588,580.11 µs (588.58 ms)
Duplicates Found in 1M dataset: 270,475
Duplicates in 50k sample: 806
Resolution Time (50k sample): 20,190.63 µs (20.19 ms)
Remaining duplicates after resolution: 0


### Timing Analysis (Linux & Python Microsecond Resolution)
1. **Linux Utilities**: Standard CLI tools include `/usr/bin/time -v` (reports wall-clock, user/sys CPU, and max RSS) and `perf`.
2. **Finest Resolution**: POSIX `clock_gettime(CLOCK_MONOTONIC)` provides nanosecond resolution without NTP clock skew.
3. **Microsecond Precision**: Python's `time.perf_counter_ns()` uses hardware monotonic performance counters (e.g. x86 TSC). Dividing nanoseconds by $1000$ provides exact microsecond ($\mu	ext{s}$) resolution.


## Assignment 4: Find Element at Index $i$ if Array Were Sorted
- **Approaches**:
  1. Priority Queue (Heap): $O(n \log k)$
  2. Quicksort Partition (Quickselect): $O(n)$ average
  3. Full Sort (Baseline): $O(n \log n)$


In [5]:
# 1. Priority Queue (Heap)
def select_heap(arr, i):
    n = len(arr)
    if i < n // 2:
        return heapq.nsmallest(i + 1, arr)[-1]
    else:
        return heapq.nlargest(n - i, arr)[-1]

# 2. Quickselect using Quicksort Partition Routine
def quickselect(arr, i):
    def partition(a, l, r):
        pivot = a[r]
        p = l
        for j in range(l, r):
            if a[j] <= pivot:
                a[p], a[j] = a[j], a[p]
                p += 1
        a[p], a[r] = a[r], a[p]
        return p

    def select(a, l, r, k):
        if l == r:
            return a[l]
        p = partition(a, l, r)
        if k == p:
            return a[p]
        elif k < p:
            return select(a, l, p - 1, k)
        else:
            return select(a, p + 1, r, k)

    return select(arr.copy(), 0, len(arr) - 1, i)

# 3. Full Sort Baseline
def select_sort(arr, i):
    return sorted(arr)[i]


In [6]:
n_arr = 50_000
test_data = [random.randint(1, 1_000_000) for _ in range(n_arr)]
test_indices = [0, 10, n_arr // 2, n_arr - 11, n_arr - 1]

print(f"{'Target i':>10} | {'Heap (µs)':>12} | {'Quickselect (µs)':>18} | {'Sort (µs)':>12} | {'All Match':>10}")
print("-" * 72)
for i in test_indices:
    val_sort, t_sort = timer(select_sort, test_data, i)
    val_heap, t_heap = timer(select_heap, test_data, i)
    val_qs, t_qs = timer(quickselect, test_data, i)
    match = (val_sort == val_heap == val_qs)
    print(f"{i:>10} | {t_heap:>12.2f} | {t_qs:>18.2f} | {t_sort:>12.2f} | {str(match):>10}")


  Target i |    Heap (µs) |   Quickselect (µs) |    Sort (µs) |  All Match
------------------------------------------------------------------------
         0 |       927.85 |           27941.30 |     13153.62 |       True
        10 |      1670.34 |           28237.52 |     14423.20 |       True
     25000 |    132636.20 |           29368.93 |     12444.46 |       True
     49989 |      1512.62 |           12418.63 |     13670.59 |       True
     49999 |       763.77 |           10999.95 |     11549.42 |       True


## Assignment 5: Cost of Execution & Asymptotic Loop Analysis
Empirical counting of statements executed ($x = x + 1$) and execution times vs theoretical complexities:
- **(a) Code 1**: $\sum_{i=1}^n i = \frac{n(n+1)}{2} = \Theta(n^2)$
- **(b) Code 2**:
  - *Paper text*: `j = n/2;` inside `while (j >= 1)` creates an infinite loop when $n \ge 2$ because $j$ never decreases.
  - *Intended algorithm*: `j = j // 2;` which halves $j$ each iteration: $\sum_{k=0}^{\lfloor \log_2 n \rfloor} \frac{n}{2^k} < 2n = \Theta(n)$.
- **(c) Code 3**: $\sum_{i=1}^n i^2 = \frac{n(n+1)(2n+1)}{6} = \Theta(n^3)$
- **(d) Code 4**: Outer loop runs $\lfloor \log_2 n \rfloor + 1$ times, inner loop $n$ times: $\Theta(n \log n)$


In [7]:
def code1(n):
    x = 0
    for i in range(1, n + 1):
        for j in range(1, i + 1):
            x += 1
    return x

def code2(n):
    x = 0
    j = n
    while j >= 1:
        for i in range(1, j + 1):
            x += 1
        j = j // 2  # Intended logarithmic halving
    return x

def code3(n):
    x = 0
    for i in range(1, n + 1):
        for j in range(1, i + 1):
            for k in range(1, i + 1):
                x += 1
    return x

def code4(n):
    x = 0
    i = n
    while i >= 1:
        for j in range(1, n + 1):
            x += 1
        i = i // 2
    return x


In [8]:
print("=== Exact Step Counts vs Theoretical Formulas ===")
for n in [10, 50, 100]:
    c1, _ = timer(code1, n)
    c2, _ = timer(code2, n)
    c3, _ = timer(code3, n)
    c4, _ = timer(code4, n)
    th1 = n * (n + 1) // 2
    th3 = n * (n + 1) * (2 * n + 1) // 6
    print(f"n={n:>3} | Code1: {c1:>6} (theo {th1}) | Code2: {c2:>4} | Code3: {c3:>7} (theo {th3}) | Code4: {c4:>5}")

print("\n=== Empirical Execution Times (µs) ===")
print(f"{'n':>5} | {'Code 1 O(n²)':>14} | {'Code 2 O(n)':>13} | {'Code 3 O(n³)':>14} | {'Code 4 O(n log n)':>18}")
print("-" * 74)
for n in [50, 100, 200, 300]:
    _, t1 = timer(code1, n)
    _, t2 = timer(code2, n)
    _, t3 = timer(code3, n)
    _, t4 = timer(code4, n)
    print(f"{n:>5} | {t1:>14.2f} | {t2:>13.2f} | {t3:>14.2f} | {t4:>18.2f}")


=== Exact Step Counts vs Theoretical Formulas ===
n= 10 | Code1:     55 (theo 55) | Code2:   18 | Code3:     385 (theo 385) | Code4:    40
n= 50 | Code1:   1275 (theo 1275) | Code2:   97 | Code3:   42925 (theo 42925) | Code4:   300
n=100 | Code1:   5050 (theo 5050) | Code2:  197 | Code3:  338350 (theo 338350) | Code4:   700

=== Empirical Execution Times (µs) ===
    n |   Code 1 O(n²) |   Code 2 O(n) |   Code 3 O(n³) |  Code 4 O(n log n)
--------------------------------------------------------------------------
   50 |          84.84 |          6.43 |        3283.11 |              22.49
  100 |         472.45 |         15.35 |       22867.57 |              42.39
  200 |        1155.02 |         18.72 |      148196.00 |              75.33
  300 |        2065.12 |         27.72 |      505136.97 |             158.19


## Assignment 6: Large Integer Multiplication (Karatsuba vs Conventional)
- **Conventional Multiplication**: Digit-by-digit grade-school multiplication: $\Theta(n^2)$.
- **Karatsuba Algorithm**: Divide and conquer splitting integers into halves and reducing 4 multiplications to 3: $\Theta(n^{\log_2 3}) \approx \Theta(n^{1.585})$.


In [6]:
# 1. Conventional Large Integer Multiplication O(n^2)
def mult_conventional(x, y):
    s_y = str(y)
    total = 0
    for i, digit in enumerate(reversed(s_y)):
        total += (x * int(digit)) * (10 ** i)
    return total

# 2. Karatsuba Divide and Conquer Multiplication O(n^1.585)
def karatsuba(x, y):
    if x < 10 or y < 10:
        return x * y
    n = max(len(str(x)), len(str(y)))
    if n <= 16:
        return x * y
    m = n // 2
    p = 10 ** m
    x1, x0 = divmod(x, p)
    y1, y0 = divmod(y, p)
    z0 = karatsuba(x0, y0)
    z2 = karatsuba(x1, y1)
    z1 = karatsuba(x0 + x1, y0 + y1) - z0 - z2
    return z2 * (10 ** (2 * m)) + z1 * p + z0


In [7]:
random.seed(42)
print(f"{'Digits':>6} | {'Conventional (µs)':>18} | {'Karatsuba (µs)':>16} | {'Speedup':>10} | {'Matches':>8}")
print("-" * 68)

for d in [32, 64, 128, 256, 512, 1024]:
    x = random.randint(10 ** (d - 1), 10 ** d - 1)
    y = random.randint(10 ** (d - 1), 10 ** d - 1)
    
    (res_conv, t_conv) = timer(mult_conventional, x, y)
    (res_kar, t_kar) = timer(karatsuba, x, y)
    
    match = (res_conv == res_kar == (x * y))
    speedup = t_conv / t_kar if t_kar > 0 else 1.0
    print(f"{d:>6} | {t_conv:>18.2f} | {t_kar:>16.2f} | {speedup:>9.2f}x | {str(match):>8}")


Digits |  Conventional (µs) |   Karatsuba (µs) |    Speedup |  Matches
--------------------------------------------------------------------
    32 |              46.43 |            21.12 |      2.20x |     True
    64 |              75.30 |            81.21 |      0.93x |     True
   128 |             167.84 |           113.15 |      1.48x |     True
   256 |             611.74 |           302.27 |      2.02x |     True
   512 |            2279.90 |           970.40 |      2.35x |     True
  1024 |           15247.29 |          2985.25 |      5.11x |     True


### Theoretical vs Empirical Complexity Analysis
- **Conventional Multiplication**: Each of the $n$ digits of $y$ is multiplied by $x$ and shifted, requiring $\Theta(n^2)$ single-digit operations.
- **Karatsuba Algorithm**: Computes $(x_1 + x_0)(y_1 + y_0) - z_0 - z_2$ to find the middle cross term with only 3 recursive multiplications: $T(n) = 3T(n/2) + O(n)$, yielding $O(n^{\log_2 3}) \approx O(n^{1.585})$.
- **Empirical Confirmation**: While conventional multiplication quadruples in time when digit length doubles ($O(n^2)$), Karatsuba scales at approximately $3\times$ per doubling, becoming over $5\times$ faster at 1,024 digits.
